# Brain_Scape — 05 LLM Pipeline

Run RAG retrieval, Q&A, and report generation using the analysis output from `04_analysis.ipynb`.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import json
from pathlib import Path

data_dir = Path('../data')
job_id = 'notebook-demo'
output_dir = data_dir / 'outputs' / job_id

## 1. Load Analysis Results

Read the analysis JSON from the previous notebook.

In [ ]:
analysis_path = output_dir / 'analysis.json'

if analysis_path.exists():
    with open(analysis_path) as f:
        analysis = json.load(f)
    print(f'Analysis loaded: {len(analysis.get("damage_summary", []))} regions')
    print(f'Overall confidence: {analysis.get("overall_confidence", 0):.3f}')
else:
    # Use synthetic data if analysis hasn't been run
    analysis = {
        'schema_version': '1.0.0',
        'scan_id': job_id,
        'damage_summary': [
            {
                'anatomical_name': 'Left Hippocampus',
                'severity_level': 4,
                'severity_label': 'RED',
                'confidence': 0.91,
                'volume_mm3': 823.4,
                'volume_pct_of_region': 61.2,
            },
            {
                'anatomical_name': 'Right Frontal Lobe',
                'severity_level': 2,
                'severity_label': 'YELLOW',
                'confidence': 0.78,
                'volume_mm3': 450.0,
                'volume_pct_of_region': 15.0,
            },
        ],
        'overall_confidence': 0.85,
        'differential_diagnosis': [],
        'connectivity': {},
        'scan_metadata': {'scan_id': job_id, 'modality': 'MRI_T1'},
    }
    print('Using synthetic analysis data (run 04_analysis.ipynb for real results)')

## 2. Prompt Template Inspection

Examine the versioned prompt templates used for all LLM interactions.

In [ ]:
from llm.prompt_templates import VERSION, TEMPLATES, qa_prompt, clinician_report_prompt

print(f'Prompt template version: {VERSION}')
print(f'Registered templates: {list(TEMPLATES.keys())}')

# Render a sample QA prompt
sample_qa = qa_prompt(
    question='Is this region related to speech?',
    context='SCAN CONTEXT:\nLeft inferior frontal gyrus: ORANGE severity',
)
print(f'\n--- Sample QA Prompt ---')
print(sample_qa[:500] + '...' if len(sample_qa) > 500 else sample_qa)

## 3. RAG Retrieval

Embed a clinical query and retrieve relevant literature from the vector store.

In [ ]:
from llm.rag_engine import RAGEngine

rag = RAGEngine(vector_store='weaviate')

# Embed a test query
query = 'hippocampal damage and memory loss'
embedding = rag.embed_text(query)
print(f'Query: "{query}"')
print(f'Embedding dimension: {len(embedding)}')
print(f'Embedding sample: {embedding[:5]}')

# Retrieve relevant documents
results = rag.retrieve(query, top_k=5)
print(f'\nRetrieved {len(results)} documents:')
for i, doc in enumerate(results, 1):
    print(f'  [{i}] {doc.get("title", "Untitled")} (score: {doc.get("score", 0):.3f})')

## 4. Q&A Engine

Ask clinical questions about the scan. Uses RAG context + scan analysis.

Without an LLM API key, the engine falls back to template-based answers.

In [ ]:
from llm.qa_engine import QAEngine

qa = QAEngine(rag_engine=rag, llm_client=None)

# Ask multiple questions
questions = [
    'What regions are damaged?',
    'Is the hippocampus affected?',
    'What is the confidence level of these findings?',
]

for q in questions:
    result = qa.answer(q, analysis)
    print(f'Q: {q}')
    print(f'A: {result["answer"]}')
    print(f'Confidence: {result.get("confidence", "N/A")}')
    if result.get('citations'):
        print(f'Citations: {len(result["citations"])} references')
    print()

## 5. Report Generation

Generate a structured clinical report. Two-pass process:
1. Structured extraction from analysis JSON
2. Clinician-mode rendering with full technical depth

Without an LLM API key, uses template-based report generation.

In [ ]:
from llm.report_generator import ReportGenerator

generator = ReportGenerator(rag_engine=rag, llm_client=None)

# Generate clinician report
report = generator.generate(
    analysis=analysis,
    mode='clinician',
    scan_id=job_id,
)

print('=== CLINICIAN REPORT ===')
print(f'\nExecutive Summary:')
print(report.get('executive_summary', 'N/A'))
print(f'\nDamage Assessment:')
for region in report.get('damage_assessment', []):
    print(f'  {region.get("anatomical_name", "?")}: {region.get("severity_label", "?")} '
          f'(confidence: {region.get("confidence", 0):.2f})')
print(f'\nOverall Confidence: {report.get("overall_confidence", "N/A")}')

## 6. PDF Export

Export the report as a PDF file using ReportLab or WeasyPrint.

In [ ]:
# Generate and save PDF
pdf_path = str(output_dir / 'clinician_report.pdf')

try:
    saved_path = generator.save_pdf(report, pdf_path)
    print(f'PDF saved: {saved_path}')
except Exception as e:
    print(f'PDF generation requires ReportLab/WeasyPrint: {e}')
    print('Report data is available in the `report` variable above.')

## 7. Full Pipeline Output Summary

All Phase 1 outputs in `data/outputs/{job_id}/`:

In [ ]:
print(f'=== Brain_Scape Pipeline Output: {job_id} ===')
print()

if output_dir.exists():
    for f in sorted(output_dir.iterdir()):
        size = f.stat().st_size
        if size < 1024:
            size_str = f'{size} B'
        elif size < 1024 * 1024:
            size_str = f'{size / 1024:.1f} KB'
        else:
            size_str = f'{size / (1024*1024):.1f} MB'
        print(f'  {f.name:40s} {size_str}')
else:
    print('  No outputs yet.')

print()
print('Phase 1 pipeline complete!')
print('Next: Deploy via docker-compose or run `bash scripts/run_pipeline.sh <scan_path>`')